# 06 — Analyse d'erreurs et recommandations

**Projet** : Prédiction d'attrition client (churn télécom) avec TensorFlow
**Principe** : une métrique globale ne dit **jamais** quoi corriger. Ce notebook descend au
niveau de la ligne : qui le modèle se trompe-t-il, avec quelle confiance, et que fait-on lundi matin ?

Le split de **test** n'est utilisé qu'ici — une seule fois — pour rester une estimation honnête.

## Objectifs pédagogiques

1. Produire une évaluation complète (métriques, matrice, courbes, calibration).
1. Arbitrer le **seuil de décision** avec une table métier (précision / rappel / volume).
1. Segmenter les erreurs pour identifier une cause actionnable.
1. Formuler des recommandations concrètes, appuyées sur les chiffres observés.

**Objectifs transverses du dépôt**

- Écrire une boucle d'entraînement TensorFlow bas niveau : tf.data, GradientTape, apply_gradients.
- Compiler le pas d'entraînement avec tf.function et comprendre le retraçage (reduce_retracing).
- Subclasser keras.Model et keras.layers.Layer : call(features, training=...), get_config().

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=12",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
# Les groupes (l'identité de l'utilisateur en classement) voyagent à côté des matrices : le
# pré-traitement supprime cette colonne, or une métrique par groupe ne peut pas s'en passer.
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    groups=PREPARED.get("groups_train"),
    groups_val=PREPARED.get("groups_val"),
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

## 1. Évaluation sur le split de test

In [ ]:
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"],
    PREPARED["y_test"],
    split="test",
    context=PREPARED["enriched"]["test"],
)

metrics_frame = pd.DataFrame(
    {"métrique": list(RESULT.metrics), "valeur": [RESULT.metrics[name] for name in RESULT.metrics]}
)
print(f"observations évaluées : {RESULT.n_samples} | taux d'erreur : {RESULT.error_rate:.2%}")
metrics_frame.round(4)

**Ce qu'il faut retenir**

- La métrique de décision est `roc_auc` = **celle affichée ci-dessus** — c'est elle qui pilote le seuil de qualité.
- Le `context` passé à `evaluate()` permet d'enrichir l'analyse d'erreurs avec les colonnes brutes (identifiants, segments).
- Comparer systématiquement au split de validation : un écart important signale un test trop petit ou une dérive.

## 2. Matrice de confusion et métriques par classe

In [ ]:
from src.visualization.plots import ClassificationPlots

PLOTS = ClassificationPlots(NB_PATHS.figures_dir)
confusion_path = PLOTS.confusion_matrix(RESULT)
display(Image(confusion_path, width=460))

In [ ]:
if not RESULT.per_class.empty:
    display(RESULT.per_class.round(4))
    per_class_path = PLOTS.per_class_metrics(RESULT.per_class)
    display(Image(per_class_path, width=520))
else:
    print("Aucune métrique par classe pour cette tâche.")

**Ce qu'il faut retenir**

- Lire la matrice **en lignes** (rappel) puis **en colonnes** (précision) : les deux erreurs ne coûtent pas la même chose.
- Une classe au rappel faible = des cas positifs non détectés ; une précision faible = du bruit envoyé aux équipes.
- Le coût métier asymétrique (faux négatif ≫ faux positif) doit se traduire dans le seuil, pas dans la matrice.

## 3. Courbes et calibration

In [ ]:
figures = {}
for name in ("roc_curve", "precision_recall_curve", "calibration_curve", "score_distribution"):
    method = getattr(PLOTS, name, None)
    if method is None:
        continue
    path = method(RESULT)
    if path is not None:
        figures[name] = path
        display(Image(path, width=470))
print("figures générées :", sorted(figures))

**Ce qu'il faut retenir**

- Le **ROC AUC** est optimiste en fort déséquilibre : le **PR AUC** reflète mieux la valeur opérationnelle.
- Une courbe de calibration éloignée de la diagonale interdit d'utiliser les probabilités comme des revenus attendus.
- La distribution des scores montre si le seuil par défaut (0.5) tombe dans une zone dense : souvent non.

## 4. Arbitrage du seuil — la table à montrer aux métiers

In [ ]:
THRESHOLDS = EVALUATOR.threshold_analysis(PREPARED["X_test"], PREPARED["y_test"])
if THRESHOLDS.empty:
    print("Pas de probabilités disponibles : l'arbitrage de seuil ne s'applique pas à ce modèle.")
else:
    display(THRESHOLDS.round(4).head(20))
    threshold_path = PLOTS.threshold_curve(THRESHOLDS)
    if threshold_path is not None:
        display(Image(threshold_path, width=560))

In [ ]:
if not THRESHOLDS.empty:
    best_f1 = THRESHOLDS.loc[THRESHOLDS["f1"].idxmax()]
    volume_column = next(
        (name for name in THRESHOLDS.columns if "volume" in name or "flag" in name), None
    )
    print(f"seuil maximisant le F1 : {float(best_f1['threshold']):.2f}")
    print(
        f"  précision = {float(best_f1['precision']):.3f} | rappel = {float(best_f1['recall']):.3f}"
    )
    if volume_column:
        print(f"  {volume_column} = {best_f1[volume_column]}")
    print("\nLecture métier : baisser le seuil augmente le volume à traiter et le rappel ;")
    print("l'augmenter réduit la charge des équipes mais laisse passer des cas positifs.")

**Ce qu'il faut retenir**

- Le seuil est un **paramètre métier** : il se choisit avec la capacité de traitement disponible, pas à 0.5 par défaut.
- Documenter la table dans le rapport (`ReportBuilder`) évite de re-débattre du seuil à chaque comité.

## 5. Où le modèle se trompe-t-il ?

In [ ]:
if RESULT.errors.empty:
    print("Aucune erreur sur le split de test.")
else:
    print(f"{len(RESULT.errors)} erreurs analysées (triées par confiance décroissante)")
    display(RESULT.errors.head(12))

In [ ]:
# Segmentation des erreurs : quelle population concentre les faux positifs / faux négatifs ?
if not RESULT.errors.empty and CONFIG.data.target:
    error_frame = RESULT.errors.copy()
    segmentation_columns = [
        column
        for column in (
            ["contract_type", "internet_service", "payment_method", "region", "has_promotion"]
        )
        if column in error_frame.columns
    ][:3]
    for column in segmentation_columns:
        grouped = (
            error_frame.groupby(error_frame[column].astype(str))
            .agg(erreurs=("is_error", "size"))
            .sort_values("erreurs", ascending=False)
        )
        total = raw.groupby(raw[column].astype(str)).size().rename("population")
        table = grouped.join(total, how="left")
        table["taux_erreur"] = (table["erreurs"] / table["population"]).round(4)
        print(f"--- erreurs par `{column}` ---")
        display(table.head(6))
else:
    print("Segmentation indisponible pour cette tâche.")

**Ce qu'il faut retenir**

- Un segment dont le taux d'erreur dépasse nettement la moyenne est une **cause**, pas une anecdote.
- Vérifier ensuite le volume de données de ce segment à l'entraînement : sous-représentation = sous-performance.
- Les réponses possibles : feature dédiée, sur-échantillonnage du segment, ou règle métier complémentaire.

In [ ]:
baseline_comparison = EVALUATOR.compare_to_baseline(PREPARED["X_test"], PREPARED["y_test"])
gains = pd.DataFrame(
    {
        "modèle": ["modèle entraîné", "baseline"],
        CONFIG.metrics.primary: [
            RESULT.metrics.get(CONFIG.metrics.primary, float("nan")),
            baseline_comparison.get(f"baseline_{CONFIG.metrics.primary}", float("nan")),
        ],
    }
)
gains.round(4)

## 6. Rapport exécutable

In [ ]:
from src.evaluation.reports import ReportBuilder

REPORTER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
WRITTEN = REPORTER.build(
    RESULT, model=MODEL, thresholds=THRESHOLDS if not THRESHOLDS.empty else None
)
print(f"{len(WRITTEN)} artefacts écrits dans {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")
report_path = WRITTEN["report"]
Markdown(report_path.read_text(encoding="utf-8")[:2500] + "\n\n[…]")

**Ce qu'il faut retenir**

- Le rapport mélange **chiffres calculés** et **recommandations documentées** : il est régénérable à chaque run.
- Le même contenu est écrit en JSON (`artifacts/metrics`) pour être consommé par un dashboard ou une CI.

In [ ]:
recommendations = REPORTER.recommendations(RESULT)
for index, recommendation in enumerate(recommendations, start=1):
    print(f"{index:2d}. {recommendation}")

## 7. Recommandations concrètes

### issues de l'analyse (calculées ci-dessus)

Elles dépendent des métriques observées : seuil à ajuster, classe à rappeler, calibration à
revoir, volume d'erreurs à investiguer.

### documentées pour ce cas d'usage

1. Piloter le modèle sur le rappel de la classe churn et le coût par alerte, pas sur l'accuracy globale.
2. Choisir le seuil de décision avec la table d'arbitrage précision/rappel/volume (voir le rapport) plutôt qu'avec 0.5 par défaut.
3. Recalibrer les probabilités (isotonic ou Platt) avant tout usage en revenu attendu ou en scoring CRM.
4. Ajouter des features temporelles (évolution de la consommation, récence du dernier ticket) : c'est le levier de performance principal.
5. Mettre en place un suivi de dérive sur `contract_type`, `monthly_charges` et `satisfaction_score` (voir mlops/model-monitoring).
6. Documenter les variables proxy (moyen de paiement) pour éviter un biais commercial involontaire.

### plan d'action proposé

| Priorité | Action | Effet attendu | Comment vérifier |
| --- | --- | --- | --- |
| 1 | Fixer le seuil avec la table d'arbitrage | volume d'alertes maîtrisé | `threshold_analysis()` rejoué |
| 2 | Recalibrer les probabilités si la diagonale est manquée | coûts/revenus attendus fiables | `calibration_curve` |
| 3 | Instrumenter les segments les plus en erreur | rappel ciblé | segmentation du §5 |
| 4 | Ajouter un suivi de dérive sur les features dominantes | alerte précoce | `mlops/model-monitoring` |
| 5 | Rejouer ce notebook à chaque nouvelle version de données | non-régression | `make evaluate` + CI |

## 8. Limites assumées

- Les données sont **synthétiques** : les niveaux de performance illustrent une méthode, pas un marché réel.
- Une seule passe d'évaluation : la variance n'est pas mesurée ici (voir notebook 05, §4).
- L'analyse d'erreurs porte sur 1500 lignes maximum pour rester interactive.

**Aller plus loin dans le dépôt** : comparaison multi-stacks (`data-science/classification/with-*`),
mise en production et suivi (`mlops/`), pipelines de données (`data-eng/`).